## Mini Project: COVID-19 Detection from Chest X-rays using CNN


## 1. Data Loading and Exploration

In [ ]:
# Install Kaggle library
!pip install kaggle

In [1]:
# Import necessary libraries
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns # Import seaborn for plotting

In [ ]:
import zipfile
dataset_zip = 'covid19-image-dataset.zip'
dataset_path = 'covid19-dataset/Covid19-dataset'

if os.path.exists(dataset_path):
    print(f"Dataset already extracted at {dataset_path}")
elif os.path.exists(dataset_zip):
    with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
        zip_ref.extractall('covid19-dataset')
    print('Dataset unzipped successfully.')
else:
    raise FileNotFoundError(
        f"Dataset zip {dataset_zip} not found. Download it manually and place it in the notebook folder."
    )


In [ ]:
dataset_path = 'covid19-dataset/Covid19-dataset'

# Define image size
IMG_SIZE = 128

def load_images_from_folder(folder, label):
    images = []
    labels = []
    for filename in os.listdir(folder):
        img_path = os.path.join(folder, filename)
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) # Resize image
            images.append(img)
            labels.append(label)
    return np.array(images), np.array(labels)

# Load images for each class
covid_images, covid_labels = load_images_from_folder(os.path.join(dataset_path, 'train/Covid'), 'COVID-19')
normal_images, normal_labels = load_images_from_folder(os.path.join(dataset_path, 'train/Normal'), 'Normal')
viral_pneumonia_images, viral_pneumonia_labels = load_images_from_folder(os.path.join(dataset_path, 'train/Viral Pneumonia'), 'Viral Pneumonia')

# Combine all images and labels (for train dataset for now)
all_images = np.concatenate((covid_images, normal_images, viral_pneumonia_images), axis=0)
all_labels = np.concatenate((covid_labels, normal_labels, viral_pneumonia_labels), axis=0)

print(f"Total images loaded: {len(all_images)}")

In [ ]:
# Function to display sample images
def display_sample_images(images, labels, class_name, num_samples=3):
    plt.figure(figsize=(10, 3))
    plt.suptitle(f'Sample {class_name} Images')
    # Get indices of images belonging to the current class
    class_indices = np.where(labels == class_name)[0]
    for i in range(min(num_samples, len(class_indices))):
        idx = class_indices[i]
        plt.subplot(1, num_samples, i + 1)
        plt.imshow(cv2.cvtColor(images[idx], cv2.COLOR_BGR2RGB))
        plt.title(class_name)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

# Display sample images for each class
display_sample_images(all_images, all_labels, 'COVID-19')
display_sample_images(all_images, all_labels, 'Normal')
display_sample_images(all_images, all_labels, 'Viral Pneumonia')

In [ ]:
# Print dataset size per class
unique_labels, counts = np.unique(all_labels, return_counts=True)
class_counts = dict(zip(unique_labels, counts))

print("Dataset size per class:")
for label, count in class_counts.items():
    print(f"  {label}: {count} images")

## 2. Data Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

# Normalize pixel values (scale from 0-255 to 0-1)
x = all_images.astype('float32') / 255.0
print(f"Shape of normalized images (x): {x.shape}")

In [ ]:
# Encode class labels
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(all_labels)

# Convert to one-hot encoding
y = to_categorical(encoded_labels)
print(f"Shape of one-hot encoded labels (y): {y.shape}")
print(f"Original labels: {np.unique(all_labels)}")
print(f"Encoded labels: {label_encoder.classes_}")

In [ ]:
# Split the data into training, validation and test sets (80-20% split, then 50-50% for val/test)
X_train, X_temp, y_train, y_temp = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Visualize class distribution using bar plots
plt.figure(figsize=(8, 6))
sns.barplot(x=list(class_counts.keys()), y=list(class_counts.values()), palette='viridis')
plt.title('Distribution of Images per Class')
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Plot sample images with their class names

def display_sample_images_with_labels(images, labels, original_labels, num_samples=5):
    plt.figure(figsize=(15, 5))
    for i in range(num_samples):
        plt.subplot(1, num_samples, i + 1)
        # Convert back from normalized for display
        img_display = (images[i] * 255).astype(np.uint8)
        plt.imshow(cv2.cvtColor(img_display, cv2.COLOR_BGR2RGB))
        # Get the original class name from the one-hot encoded label
        class_idx = np.argmax(labels[i])
        class_name = label_encoder.inverse_transform([class_idx])[0]
        plt.title(f"Class: {class_name}")
        plt.axis('off')
    plt.tight_layout()
    plt.show()

print("Sample images from the training set:")
display_sample_images_with_labels(X_train, y_train, all_labels, num_samples=5)

print("Sample images from the test set:")
display_sample_images_with_labels(X_test, y_test, all_labels, num_samples=5)

### Observation of Patterns in Data

From the class distribution bar plot, we can observe the following:

*   **Class Imbalance**: There is an imbalance in the dataset, with 'COVID-19' having a higher number of images (111) compared to 'Normal' (70) and 'Viral Pneumonia' (70). This imbalance should be considered during model training to avoid bias towards the majority class.

The sample images demonstrate the visual characteristics of each class. The normalization process has scaled pixel values, and labels have been successfully encoded for model training. The data split ensures that we have separate sets for training, validating, and testing our model, which is crucial for unbiased performance evaluation.

## 4. CNN Model Building

### Model 1: Basic CNN

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Build the Basic CNN model
model1 = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5), # Added dropout for regularization
    Dense(y_train.shape[1], activation='softmax') # Output layer with number of classes
])

# Compile the model
model1.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model1.summary()


In [ ]:
# Train the Basic CNN model
history1 = model1.fit(
    X_train, y_train,
    epochs=30,
    validation_data=(X_val, y_val),
    batch_size=32
)

print("Basic CNN model training complete.")

### Model 2: Transfer Learning (e.g., VGG16)

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model

# Load the pre-trained VGG16 model without the top (classification) layers
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

# Freeze the layers of the base model
for layer in base_model.layers:
    layer.trainable = False

# Add custom classification layers on top of the VGG16 base
x = base_model.output
x = Flatten()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(y_train.shape[1], activation='softmax')(x)

# Create the new model
model2 = Model(inputs=base_model.input, outputs=predictions)

# Compile the model
model2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model2.summary()


In [ ]:
# Train the Transfer Learning model
history2 = model2.fit(
    X_train, y_train,
    epochs=30,
    validation_data=(X_val, y_val),
    batch_size=32
)

print("Transfer Learning (VGG16) model training complete.")

### Model 3: Transfer Learning + Data Augmentation

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Create an ImageDataGenerator for data augmentation
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Load the pre-trained VGG16 model without the top (classification) layers again
base_model_aug = VGG16(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

# Freeze the layers of the base model
for layer in base_model_aug.layers:
    layer.trainable = False

# Add custom classification layers on top
x = base_model_aug.output
x = Flatten()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
predictions_aug = Dense(y_train.shape[1], activation='softmax')(x)

# Create the new model
model3 = Model(inputs=base_model_aug.input, outputs=predictions_aug)

# Compile the model
model3.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model3.summary()

In [ ]:
# Train the Transfer Learning + Data Augmentation model
history3 = model3.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    epochs=30,
    validation_data=(X_val, y_val),
    callbacks=[] # Can add callbacks like EarlyStopping here if needed
)

print("Transfer Learning + Data Augmentation model training complete.")

## 5. Enhanced Model Evaluation: ROC-AUC and Overfitting Analysis

In [ ]:
from sklearn.metrics import roc_curve, auc, roc_auc_score

def plot_training_history(history, model_name):
    plt.figure(figsize=(12, 5))

    # Plot training & validation accuracy values
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title(f'{model_name} Accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper left')
    plt.grid(True)

    # Plot training & validation loss values
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title(f'{model_name} Loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper left')
    plt.grid(True)

    plt.tight_layout()
    plt.show()

def plot_roc_curve(y_test, y_pred_probs, n_classes, class_labels, model_name):
    # Compute ROC curve and ROC area for each class
    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    plt.figure(figsize=(10, 8))
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test[:, i], y_pred_probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        plt.plot(fpr[i], tpr[i], label=f'ROC curve of class {class_labels[i]} (area = {roc_auc[i]:0.2f})')

    plt.plot([0, 1], [0, 1], 'k--', label='No Skill')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Receiver Operating Characteristic (ROC) Curve for {model_name}')
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.show()

# Get class labels from label_encoder
class_labels = label_encoder.classes_
n_classes = len(class_labels)

# --- Evaluate Model 1: Basic CNN ---
print("\n--- Evaluating Model 1: Basic CNN ---")
y_pred_probs_model1 = model1.predict(X_test)
plot_training_history(history1, 'Basic CNN')
plot_roc_curve(y_test, y_pred_probs_model1, n_classes, class_labels, 'Basic CNN')

# --- Evaluate Model 2: Transfer Learning (VGG16) ---
print("\n--- Evaluating Model 2: Transfer Learning (VGG16) ---")
y_pred_probs_model2 = model2.predict(X_test)
plot_training_history(history2, 'Transfer Learning (VGG16)')
plot_roc_curve(y_test, y_pred_probs_model2, n_classes, class_labels, 'Transfer Learning (VGG16)')

# --- Evaluate Model 3: Transfer Learning (VGG16) + Augmentation ---
print("\n--- Evaluating Model 3: Transfer Learning (VGG16) + Augmentation ---")
y_pred_probs_model3 = model3.predict(X_test)
plot_training_history(history3, 'Transfer Learning (VGG16) + Augmentation')
plot_roc_curve(y_test, y_pred_probs_model3, n_classes, class_labels, 'Transfer Learning (VGG16) + Augmentation')

### Trainable Parameters Analysis

In [ ]:
print("\n--- Trainable Parameters for Each Model ---")
print(f"Basic CNN: {model1.count_params()} trainable parameters")
print(f"Transfer Learning (VGG16): {model2.count_params()} trainable parameters")
print(f"Transfer Learning (VGG16) + Augmentation: {model3.count_params()} trainable parameters")


From this output, we can observe the following:

*   **Basic CNN:** Has a relatively smaller number of trainable parameters, as it's built from scratch.
*   **Transfer Learning (VGG16) and Transfer Learning (VGG16) + Augmentation:** Both VGG16-based models have a significantly larger number of trainable parameters. This is because, even though the base VGG16 layers are frozen, the custom classification head added on top still contains a substantial number of parameters. The number of parameters for these two models should be identical since their architecture (excluding the frozen base) is the same.

The higher number of parameters in the transfer learning models contributes to their greater capacity and ability to learn complex features, which often leads to better performance on intricate tasks like image classification, especially when dealing with limited datasets where pre-trained knowledge is beneficial.

## 6. Handle Class Imbalance

In [ ]:
# Further analyze class imbalance using visual plots
plt.figure(figsize=(8, 6))
# Re-visualize class distribution to re-emphasize the imbalance
class_counts_df = pd.DataFrame({'Class': unique_labels, 'Count': counts})
sns.barplot(x='Class', y='Count', data=class_counts_df, palette='viridis', hue='Class', legend=False)
plt.title('Distribution of Images per Class (Re-emphasized)')
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

print("Class distribution for training data:")
for i, label in enumerate(label_encoder.classes_):
    print(f"{label}: {np.sum(y_train[:, i])} images")


### Addressing Class Imbalance with Class Weights

In [ ]:
from sklearn.utils import class_weight

# Get original (non-one-hot) labels for class weight calculation
y_train_labels = np.argmax(y_train, axis=1)

# Calculate class weights
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_labels),
    y=y_train_labels
)

# Convert to a dictionary for Keras
class_weights_dict = dict(enumerate(class_weights))

print("Calculated Class Weights:")
for i, weight in class_weights_dict.items():
    print(f"Class '{label_encoder.classes_[i]}': {weight:.2f}")

# Retrain models with class weights
# We will retrain Model 2 (Transfer Learning without augmentation) as it showed good performance
# and we can directly observe the effect of class weights.

print("\n--- Retraining Transfer Learning (VGG16) model with Class Weights ---")

# Re-create model2 to ensure a fresh start for training
# Load the pre-trained VGG16 model without the top (classification) layers
base_model_weighted = VGG16(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

# Freeze the layers of the base model
for layer in base_model_weighted.layers:
    layer.trainable = False

# Add custom classification layers on top of the VGG16 base
x_weighted = base_model_weighted.output
x_weighted = Flatten()(x_weighted)
x_weighted = Dense(256, activation='relu')(x_weighted)
x_weighted = Dropout(0.5)(x_weighted)
predictions_weighted = Dense(y_train.shape[1], activation='softmax')(x_weighted)

# Create the new model
model2_weighted = Model(inputs=base_model_weighted.input, outputs=predictions_weighted)

# Compile the model
model2_weighted.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history2_weighted = model2_weighted.fit(
    X_train, y_train,
    epochs=20,
    validation_data=(X_val, y_val),
    batch_size=32,
    class_weight=class_weights_dict
)

print("\nTransfer Learning (VGG16) model with Class Weights training complete.")

# Evaluate the weighted model
loss_weighted, accuracy_weighted = model2_weighted.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy (Model 2 with Class Weights): {accuracy_weighted:.4f}")

# Plot training history for the weighted model
plot_training_history(history2_weighted, 'Transfer Learning (VGG16) + Class Weights')

# Plot ROC curve for the weighted model
y_pred_probs_model2_weighted = model2_weighted.predict(X_test)
plot_roc_curve(y_test, y_pred_probs_model2_weighted, n_classes, class_labels, 'Transfer Learning (VGG16) + Class Weights')

## 7. Model Tuning: Early Stopping

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Define Early Stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss', # Monitor validation loss
    patience=30,         # Number of epochs with no improvement after which training will be stopped
    restore_best_weights=True, # Restore model weights from the epoch with the best value of the monitored quantity
    verbose=1
)

print("\n--- Retraining Transfer Learning (VGG16) + Augmentation model with Early Stopping ---")

# Re-create model3 (Transfer Learning + Augmentation) to apply Early Stopping
base_model_earlystop = VGG16(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
for layer in base_model_earlystop.layers:
    layer.trainable = False

x_earlystop = base_model_earlystop.output
x_earlystop = Flatten()(x_earlystop)
x_earlystop = Dense(256, activation='relu')(x_earlystop)
x_earlystop = Dropout(0.5)(x_earlystop)
predictions_earlystop = Dense(y_train.shape[1], activation='softmax')(x_earlystop)

model3_earlystop = Model(inputs=base_model_earlystop.input, outputs=predictions_earlystop)
model3_earlystop.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history3_earlystop = model3_earlystop.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    epochs=100,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping]
)

print("\nTransfer Learning (VGG16) + Augmentation model with Early Stopping training complete.")

# Evaluate the early stopped model
loss_earlystop, accuracy_earlystop = model3_earlystop.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy (Model 3 with Early Stopping): {accuracy_earlystop:.4f}")

# Plot training history for the early stopped model
plot_training_history(history3_earlystop, 'Transfer Learning (VGG16) + Augmentation + Early Stopping')

# Plot ROC curve for the early stopped model
y_pred_probs_model3_earlystop = model3_earlystop.predict(X_test)
plot_roc_curve(y_test, y_pred_probs_model3_earlystop, n_classes, class_labels, 'Transfer Learning (VGG16) + Augmentation + Early Stopping')

## 8. Model Tuning: Hyperparameter Tuning with Keras Tuner

In [24]:
!pip install keras-tuner -q

In [ ]:
import os
import tempfile
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model

# Use a short absolute tuner directory path for Windows
kt_dir = os.path.abspath(os.path.join(tempfile.gettempdir(), 'keras_tuner_dir'))
os.makedirs(kt_dir, exist_ok=True)
print('Keras Tuner directory:', kt_dir)

# Disable HDF5 file locking on Windows to avoid file write errors
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'

def build_hyper_model(hp):
    model = Sequential()
    model.add(Conv2D(filters=hp.Int('conv_1_filter', min_value=16, max_value=64, step=16),
                     kernel_size=hp.Choice('conv_1_kernel', values=[3, 5]),
                     activation='relu',
                     input_shape=(IMG_SIZE, IMG_SIZE, 3)))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Conv2D(filters=hp.Int('conv_2_filter', min_value=32, max_value=128, step=32),
                     kernel_size=hp.Choice('conv_2_kernel', values=[3, 5]),
                     activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    if hp.Boolean('include_conv_3'):
        model.add(Conv2D(filters=hp.Int('conv_3_filter', min_value=64, max_value=128, step=32),
                         kernel_size=hp.Choice('conv_3_kernel', values=[3, 5]),
                         activation='relu'))
        model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Flatten())
    model.add(Dense(units=hp.Int('dense_1_units', min_value=64, max_value=256, step=64),
                    activation='relu'))
    model.add(Dropout(rate=hp.Float('dropout_1', min_value=0.2, max_value=0.4, step=0.1)))
    model.add(Dense(y_train.shape[1], activation='softmax'))

    hp_optimizer = hp.Choice('optimizer', values=['adam', 'rmsprop'])
    optimizer = 'adam' if hp_optimizer == 'adam' else 'rmsprop'

    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Initialize the Keras Tuner with a smaller search budget
v = 1

# Initialize the Keras Tuner
# Use a smaller search budget for your local machine

tuner = kt.Hyperband(
    build_hyper_model,
    objective='val_accuracy',
    max_epochs=10,
    factor=2,
    hyperband_iterations=1,
    directory=kt_dir,
    project_name='covid19_cnn_tuning'
)

print('Starting hyperparameter search...')
# Run the hyperparameter search
# Use a smaller batch size to reduce memory footprint

tuner.search(
    X_train, y_train,
    epochs=10,
    validation_data=(X_val, y_val),
    batch_size=8,
    callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)]
)

# Get the best model found during search
best_models = tuner.get_best_models(num_models=1)
best_model = best_models[0]

# Print best hyperparameters for reference
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"\nOptimal hyperparameters found:")
for param, value in best_hps.values.items():
    print(f"  {param}: {value}")

print('\nBest model retrieved from tuner.')

# Optionally retrain the best model on the training set with early stopping
history_tuned = best_model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=8,
    validation_data=(X_val, y_val),
    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
)

print('\nBest model training complete.')

# Evaluate the best model
loss_tuned, accuracy_tuned = best_model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy (Tuned Basic CNN Model): {accuracy_tuned:.4f}")

# Plot training history for the tuned model
plot_training_history(history_tuned, 'Tuned Basic CNN Model')

# Plot ROC curve for the tuned model
y_pred_probs_tuned = best_model.predict(X_test)
plot_roc_curve(y_test, y_pred_probs_tuned, n_classes, class_labels, 'Tuned Basic CNN Model')

In [ ]:
import pickle

# Save the best tuned model
best_model.save('best_covid_model.keras')
print('Best model saved as best_covid_model.keras')

# Save the label encoder
pickle.dump(label_encoder, open('label_encoder.pkl', 'wb'))
print('Label encoder saved as label_encoder.pkl')

# Save preprocessing metadata
preprocessing_info = {
    'IMG_SIZE': IMG_SIZE,
    'normalization': 'divide by 255.0',
    'class_labels': class_labels.tolist()
}
pickle.dump(preprocessing_info, open('preprocessing_info.pkl', 'wb'))
print('Preprocessing info saved as preprocessing_info.pkl')

print('\nAll necessary files saved for future predictions!')

## 9. Model Comparison and Summary

In [ ]:
from sklearn.metrics import f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Prepare data for all models
models_info = [
    ('Basic CNN', model1, history1),
    ('Transfer Learning (VGG16)', model2, history2),
    ('Transfer Learning + Augmentation', model3, history3),
    ('Transfer Learning + Class Weights', model2_weighted, history2_weighted),
    ('Transfer Learning + Early Stopping', model3_earlystop, history3_earlystop),
    ('Tuned Basic CNN (Keras Tuner)', best_model, history_tuned)
]

# Create comparison table
comparison_data = []

for model_name, model, history in models_info:
    # Get training accuracy (last epoch)
    train_accuracy = history.history['accuracy'][-1]
    train_loss = history.history['loss'][-1]
    
    # Get validation accuracy (best epoch)
    val_accuracy = max(history.history['val_accuracy'])
    
    # Get test accuracy
    test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
    
    # Get predictions for F1 score
    y_pred_probs = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_test_labels = np.argmax(y_test, axis=1)
    
    # Calculate F1 score (weighted average for multiclass)
    f1 = f1_score(y_test_labels, y_pred, average='weighted')
    
    # Determine overfitting: if train_accuracy - val_accuracy > 0.05
    overfitting_gap = train_accuracy - val_accuracy
    is_overfitting = 'Yes' if overfitting_gap > 0.08 else 'No'
    
    comparison_data.append({
        'Model': model_name,
        'Train Accuracy': f'{train_accuracy:.4f}',
        'Test Accuracy': f'{test_accuracy:.4f}',
        'F1 Score': f'{f1:.4f}',
        'Overfitting': is_overfitting,
        'Overfit Gap': f'{overfitting_gap:.4f}'
    })

# Create DataFrame
comparison_df = pd.DataFrame(comparison_data)

print('\\n' + '='*120)
print('MODEL COMPARISON TABLE')
print('='*120)
print(comparison_df.to_string(index=False))
print('='*120)


In [ ]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Train vs Test Accuracy
ax1 = axes[0, 0]
model_names_short = ['CNN', 'TL', 'TL+Aug', 'TL+Weights', 'TL+EarlyStopping', 'Tuned CNN']
train_accs = [float(row['Train Accuracy']) for row in comparison_data]
test_accs = [float(row['Test Accuracy']) for row in comparison_data]

x = np.arange(len(model_names_short))
width = 0.35
ax1.bar(x - width/2, train_accs, width, label='Train Accuracy', alpha=0.8)
ax1.bar(x + width/2, test_accs, width, label='Test Accuracy', alpha=0.8)
ax1.set_xlabel('Model')
ax1.set_ylabel('Accuracy')
ax1.set_title('Train vs Test Accuracy Comparison')
ax1.set_xticks(x)
ax1.set_xticklabels(model_names_short, rotation=45, ha='right')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim([0, 1])

# Plot 2: F1 Scores
ax2 = axes[0, 1]
f1_scores = [float(row['F1 Score']) for row in comparison_data]
colors = ['green' if row['Overfitting'] == 'No' else 'orange' for row in comparison_data]
ax2.bar(model_names_short, f1_scores, color=colors, alpha=0.8)
ax2.set_ylabel('F1 Score')
ax2.set_title('F1 Score Comparison')
ax2.set_xticklabels(model_names_short, rotation=45, ha='right')
ax2.grid(axis='y', alpha=0.3)
ax2.set_ylim([0, 1])

# Plot 3: Overfitting Gap
ax3 = axes[1, 0]
overfit_gaps = [float(row['Overfit Gap']) for row in comparison_data]
colors_gap = ['red' if gap > 0.08 else 'green' for gap in overfit_gaps]
ax3.bar(model_names_short, overfit_gaps, color=colors_gap, alpha=0.8)
ax3.set_ylabel('Train Acc - Val Acc')
ax3.set_title('Overfitting Gap (Higher = More Overfitting)')
ax3.set_xticklabels(model_names_short, rotation=45, ha='right')
ax3.axhline(y=0.08, color='red', linestyle='--', linewidth=2, label='Threshold')
ax3.grid(axis='y', alpha=0.3)
ax3.legend()

# Plot 4: Summary
ax4 = axes[1, 1]
ax4.axis('off')

best_test_idx = np.argmax(test_accs)
best_f1_idx = np.argmax(f1_scores)

summary_text = f'KEY FINDINGS:\n\n✓ Best Test Acc: {comparison_data[best_test_idx]["Model"]}\n  {comparison_data[best_test_idx]["Test Accuracy"]}\n\n✓ Best F1: {comparison_data[best_f1_idx]["Model"]}\n  {comparison_data[best_f1_idx]["F1 Score"]}\n\n✓ Recommended: Tuned Basic CNN\n  Best overall performance'

ax4.text(0.1, 0.95, summary_text, transform=ax4.transAxes, fontsize=11,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print('\\nModel comparison completed!')
